In [8]:
import json, pandas as pd
from clean_main_file import CleanMainFile
from calculate_productivity import CalculateProductivity

print("Starting the script")

with open("initial_parameters.json", "r") as infile:
    data = json.load(infile)


# extract the required data
emails_to_delete = data["EMAILS_TO_DELETE"]
year = data["YEAR"]
month = data["MONTH"]
coefficients = data["COEFFICIENTS"]

year_month = f"{year}-{str(month).zfill(2)}"
path = f"C:\\Productividad\\{year_month}\\"

autodesk_filename = path + "Autodesk.xlsx"

Starting the script


In [20]:
import datetime
import re
from calendar import monthrange

df_autodesk = pd.read_excel(autodesk_filename, sheet_name='Uso', usecols=[4,9])
df_autodesk['email'] = [str(email).lower() for email in df_autodesk['email']]
df_autodesk.rename(columns={'email': 'Email'}, inplace=True)
df_autodesk_table = pd.pivot_table(df_autodesk, index='Email', columns='day_used', aggfunc=len, fill_value=0)
df_autodesk_table[df_autodesk_table > 0] = 1
df_autodesk_table.columns = [str_date.strftime('%Y-%m-%d') for str_date in df_autodesk_table.columns]

columns_month = set(map(lambda day: f"{year}-{str(month).zfill(2)}-{str(day).zfill(2)}", range(1, monthrange(year, month)[1] + 1)))
set_columns = set(df_autodesk_table.columns)
columns_to_delete = set_columns - columns_month
df_autodesk_table.drop(columns=list(columns_to_delete), inplace=True)

In [24]:
input_filename = path + "data_cleaned.xlsx"
excel_file = pd.ExcelFile(input_filename)

In [4]:
import pandas as pd
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import datetime

# 1. Authenticate with your JSON key
SCOPES = ['https://www.googleapis.com/auth/admin.reports.usage.readonly']
KEY_FILE_LOCATION = 'employees-connectivity-9ad941b70bd0.json' # Update this path!
ADMIN_EMAIL = 'alejandroforero@ingetec.com.co' # Must be a Workspace Super Admin

credentials = service_account.Credentials.from_service_account_file(
    KEY_FILE_LOCATION, scopes=SCOPES)
# Delegate as the admin
delegated_credentials = credentials.with_subject(ADMIN_EMAIL)

# 2. Connect to the Admin SDK Reports API
service = build('admin', 'reports_v1', credentials=delegated_credentials)

# 3. Calculate the date — Reports API has a 2-3 day delay, so start 3 days back
#    and automatically try older dates if data is not yet available.
PARAMETERS = 'gmail:num_emails_sent,accounts:last_login_time,drive:num_items_edited,drive:num_items_viewed,drive:num_items_created'

results = None
for days_back in range(10, 14):  # Try from 3 to 7 days ago
    target_date = (datetime.datetime.now() - datetime.timedelta(days=days_back)).strftime('%Y-%m-%d')
    print(f"Trying to fetch report for {target_date}...")
    try:
        results = service.userUsageReport().get(
            userKey='all',
            date=target_date,
            parameters=PARAMETERS
        ).execute()
        print(f"✓ Successfully fetched report for {target_date}")
        break  # Data found, stop trying older dates
    except HttpError as e:
        if e.resp.status == 400 and 'not yet available' in str(e):
            print(f"  ✗ Data for {target_date} is not yet available, trying an older date...")
        else:
            raise  # Re-raise unexpected errors

if results is None:
    print("ERROR: Could not find available data for the last 7 days.")
    exit(1)

warnings = results.get('warnings', [])
if warnings:
    print("Warnings (data might not be fully ready yet):", warnings)

usage_data = results.get('usageReports', [])

# 5. Parse the data into a list of dictionaries
rows = []
for report in usage_data:
    email = report.get('entity', {}).get('userEmail')
    
    # helper to safely extract metrics
    def get_metric(report_dict, key):
        metrics = report_dict.get('parameters', [])
        for p in metrics:
            if p.get('name') == key:
                if 'intValue' in p:
                    return int(p['intValue'])
                elif 'datetimeValue' in p:
                    return p['datetimeValue']
        return 0

    rows.append({
        'Email': email,
        'Username': email.split('@')[0],  # Basic assumption
        'Sent emails': get_metric(report, 'gmail:num_emails_sent'),
        'Last login': get_metric(report, 'accounts:last_login_time'),
        'Edited files': get_metric(report, 'drive:num_items_edited'),
        'Viewed files': get_metric(report, 'drive:num_items_viewed'),
        'Added files': get_metric(report, 'drive:num_items_created'),
    })

# 6. Append to Excel file as a new day-sheet
df = pd.DataFrame(rows)

Trying to fetch report for 2026-02-20...
✓ Successfully fetched report for 2026-02-20
Warnings (data might not be fully ready yet): [{'code': 'PARTIAL_DATA_AVAILABLE', 'message': 'Data for date 2026-02-20 for application accounts is not available right now, please try again after a few hours.', 'data': [{'key': 'date', 'value': '2026-02-20'}, {'key': 'application', 'value': 'accounts'}]}]


In [5]:
df

,Email,Username,Sent emails,Last login,Edited files,Viewed files,Added files
0,07911-drummond@ingetec.com.co,07911-drummond,0,2025-09-17T21:42:57.000Z,0,0,0
1,0808801@ingetec.com.co,0808801,0,1970-01-01T00:00:00.000Z,0,0,0
2,0808807_apoyo_componente@ingetec.com.co,0808807_apoyo_componente,0,1970-01-01T00:00:00.000Z,0,0,0
3,aamezquita@ingetec.com.co,aamezquita,10,2026-02-20T12:33:48.000Z,4,31,3
4,abautista@ingetec.com.co,abautista,3,2026-02-20T17:16:19.000Z,30,48,147
...,...,...,...,...,...,...,...
995,rojomejia@ingetec.com.co,rojomejia,0,2025-10-31T14:51:10.000Z,0,0,0
996,rolandruiz@ingetec.com.co,rolandruiz,4,2026-02-20T22:52:14.000Z,2,17,4
997,romanpineda@ingetec.com.co,romanpineda,0,2026-02-20T15:35:14.000Z,0,0,20
998,ronaldguzman@ingetec.com.co,ronaldguzman,2,2026-02-20T13:01:32.000Z,1,1,0


In [ ]:
import pandas as pd
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import datetime
from calendar import monthrange

# ── Config ───────────────────────────────────────────────────────────────
KEY_FILE_LOCATION = 'employees-connectivity-9ad941b70bd0.json'
ADMIN_EMAIL       = 'alejandroforero@ingetec.com.co'

# ⚠️ ONLY the audit scope — this is all Activity Reports needs.
#    Do NOT mix with usage.readonly (that scope is for cell 4's userUsageReport).
SCOPES = ['https://www.googleapis.com/auth/admin.reports.audit.readonly']

# ── Auth  (separate from cell 4 — different scope) ──────────────────────
credentials = service_account.Credentials.from_service_account_file(
    KEY_FILE_LOCATION, scopes=SCOPES)
delegated_credentials = credentials.with_subject(ADMIN_EMAIL)
service = build('admin', 'reports_v1', credentials=delegated_credentials)

# ── Date setup ───────────────────────────────────────────────────────────
last_day    = monthrange(year, month)[1]
month_start = f"{year}-{str(month).zfill(2)}-01T00:00:00.000Z"
month_end   = f"{year}-{str(month).zfill(2)}-{str(last_day).zfill(2)}T23:59:59.000Z"

print(f"Month window : {month_start[:10]}  →  {month_end[:10]}")
print(f"Scope        : admin.reports.audit.readonly")
print()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Fetch Chat Activity Reports  (applicationName='chat')
# Each API record = one chat event. We group by user + date → binary flag.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Fetching chat activity reports...")

activity_rows = []
page_token    = None

try:
    while True:
        kwargs = dict(
            userKey         = 'all',
            applicationName = 'chat',
            startTime       = month_start,
            endTime         = month_end,
        )
        if page_token:
            kwargs['pageToken'] = page_token

        result = service.activities().list(**kwargs).execute()
        items  = result.get('items', [])

        for item in items:
            actor     = item.get('actor', {}).get('email', '').lower()
            timestamp = item.get('id',    {}).get('time', '')
            if actor and timestamp:
                activity_rows.append({'Email': actor, 'date': timestamp[:10]})

        page_token = result.get('nextPageToken')
        if not page_token:
            break

    print(f"✓  {len(activity_rows)} chat events fetched for {month_start[:7]}")

except HttpError as e:
    print(f"✗  HTTP {e.resp.status}: {e}")
    activity_rows = []

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Build pivot table  (same format as df_autodesk_table)
# rows = Email,  columns = YYYY-MM-DD,  values = 0 or 1 (binary)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if activity_rows:
    df_events  = pd.DataFrame(activity_rows)
    df_grouped = (df_events
                  .groupby(['Email', 'date'])
                  .size()
                  .reset_index(name='messages_sent'))

    df_chats_table = df_grouped.pivot_table(
        index='Email', columns='date', values='messages_sent',
        aggfunc='sum', fill_value=0
    )
    df_chats_table[df_chats_table > 0] = 1  # binary

    # Keep only dates in the selected month
    month_dates = {
        f"{year}-{str(month).zfill(2)}-{str(d).zfill(2)}"
        for d in range(1, last_day + 1)
    }
    keep = sorted([c for c in df_chats_table.columns if c in month_dates])
    df_chats_table = df_chats_table[keep]

    print(f"\nPivot table shape : {df_chats_table.shape}")
    print(f"Employees with ≥1 active day : {(df_chats_table.sum(axis=1) > 0).sum()}")
    print()
    print(df_chats_table.iloc[:5, :5])
else:
    print("\nNo chat data retrieved.")

In [ ]:
import json

KEY_FILE_LOCATION = 'employees-connectivity-9ad941b70bd0.json'

with open(KEY_FILE_LOCATION) as f:
    key_data = json.load(f)

print("Service account email :", key_data['client_email'])
print("Client ID             :", key_data['client_id'])
print()
print("This client_id MUST match what is shown in:")
print("Admin Console → Security → API controls → Domain-wide Delegation")